# DS5 — Game Theory (GaT) Model

**Stage 4, Step 1 — Theoretical and Conceptual Modeling**
**Dataset:** DS5 (German Adolescents RCT, N=573)
**Model family:** Game Theory (GaT)
**Date:** 2026-08-06

---

## Scope and purpose

Game Theory models choices, strategies, and payoffs in repeated decision situations
(paper §2.6). In the One Sec context, every app-opening attempt is a repeated conflict
between **immediate reward** (open the app) and **reflective restraint** (dismiss, self-regulate).

This notebook implements DS5's GaT Step 1 model:
1. **Commitment score** — proportion of friction events resolved by dismissal/close (not open)
2. **Strategy classification** — commitment-oriented / reward-oriented / chronic-override / adaptive-switcher
3. **Adaptation** — does commitment change over time? (early vs. late observation window)
4. **Condition effect** — does friction shift the strategy distribution?
5. **Phenotype overlay** — do DS5_01 Profile A/B differ on GaT strategies?

### FLAG decisions

| Flag | Decision |
|---|---|
| FLAG G1 | commitment_score = rate_not_opened (dismiss + close rate) — no formal utility function |
| FLAG G2 | Strategy thresholds are descriptive, not derived from equilibrium conditions |
| FLAG G3 | No observed payoffs — strategy proxies from behavioral outcomes only |
| FLAG G4 | Adaptation measured by calendar-third change, not sequential Bayesian updating |

### Theory-to-data mapping

| Theoretical concept | Proxy variable | Limitation |
|---|---|---|
| Immediate reward | opening the app (outcome=opened) | behavioral proxy, not measured utility |
| Reflective restraint | dismissing/closing (outcome=dismissed/closed) | commitment is behavioral, not motivational |
| Strategy | commitment_score distribution + temporal pattern | no formal strategy space |
| Payoff | not directly measured | only behavioral outcomes observed |
| Adaptation | open_rate_change_calendar | calendar-period difference, not sequential update |

**Date:** 2026-08-06
**Depends on:** `outputs/features/ds5/ds5_behavior_features.csv`,
`outputs/model_outputs/stochastic_processes/participant_parameters.csv`,
`outputs/model_outputs/phenotyping/ds5_cluster_assignments.csv`
**Produces:** `outputs/model_outputs/game_theory/participant_gat_parameters.csv`,
companion `.md`, figures, model run summary


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from datetime import datetime

ROOT    = Path('..').resolve()
FEAT    = ROOT / 'outputs' / 'features' / 'ds5'
PHENO   = ROOT / 'outputs' / 'model_outputs' / 'phenotyping'
STOP_OUT = ROOT / 'outputs' / 'model_outputs' / 'stochastic_processes'
GAT_OUT = ROOT / 'outputs' / 'model_outputs' / 'game_theory'
GAT_OUT.mkdir(parents=True, exist_ok=True)
FIGS    = GAT_OUT / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

MODEL_RUN_TS = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
RNG_SEED     = 42

COMMITMENT_HI = 0.60   # > 60% not-opened → commitment-oriented
COMMITMENT_LO = 0.30   # < 30% not-opened → reward-oriented
ADAPTATION_THRESHOLD = 0.10  # |change| > 10pp considered meaningful adaptation

print('ROOT    :', ROOT)
print('GAT_OUT :', GAT_OUT)
print('Run at  :', MODEL_RUN_TS)


## Section 1: Theory — Game Theory in CAS (§2.6)

Game Theory frames the open/dismiss decision as a **repeated strategic choice** where:

- **Action A (open):** gives immediate access to target app — modeled as immediate reward
- **Action D (dismiss/close):** supports longer-term self-regulation goal — modeled as reflective restraint

The key payoff tension: opening gives immediate relief but reinforces habit. Dismissing supports
goal-directed behavior but requires overriding the immediate impulse. Friction (breathing delay)
increases the *cost* of opening by adding time and reflection, potentially shifting the strategy.

**Strategy phenotypes (paper §2.6 Example B):**
- *Commitment-oriented:* frequently dismiss; friction aligns with their strategy
- *Reward-oriented:* frequently open; friction is resisted or bypassed
- *Chronic overriders:* consistently open even under friction (long runs of openings)
- *Adaptive switchers:* modify behavior after repeated outcomes; trajectory matters

**FLAG G1:** No formal payoff matrix is constructed — no utility quantities are directly observed.
The commitment score serves as a behavioral proxy for the dismiss-preference parameter in a
repeated-game framing. Treat this as behavioral description, not as a confirmed utility estimate.


In [ ]:
beh_feat = pd.read_csv(FEAT / 'ds5_behavior_features.csv')
stop_params = pd.read_csv(STOP_OUT / 'participant_parameters.csv')
clust_v1 = pd.read_csv(PHENO / 'ds5_cluster_assignments.csv')

# Merge on participant_id_canonical
pop = beh_feat.merge(
    stop_params[['participant_id_canonical','sig_hat','mu_hat','x_mean','n_days']],
    on='participant_id_canonical', how='left'
).merge(
    clust_v1[['participant_id_canonical','profile_p1']],
    on='participant_id_canonical', how='left'
)

print(f'Total participants: {len(pop)}')
print(f'With StoP params: {pop["sig_hat"].notna().sum()}')
print(f'With phenotype: {pop["profile_p1"].notna().sum()}')
print()
print('study_condition distribution:')
print(pop['study_condition'].value_counts())


## Section 2: Commitment Score and Strategy Classification

In [ ]:
# commitment_score = fraction of friction events NOT opened (dismiss + close)
# FLAG G1: rate_not_opened = behavioral proxy for reflective restraint tendency
pop['commitment_score'] = pop['rate_not_opened']

# Strategy classification (FLAG G2: descriptive thresholds, not equilibrium-derived)
def classify_strategy(row):
    cs       = row['commitment_score']
    run_len  = row['max_consecutive_opens'] if pd.notna(row.get('max_consecutive_opens')) else 0
    adapt    = row['open_rate_change_calendar'] if pd.notna(row.get('open_rate_change_calendar')) else 0
    sig_hat  = row['sig_hat'] if pd.notna(row.get('sig_hat')) else np.nan

    # Chronic override: very low commitment AND long consecutive open streaks
    if cs < 0.20 and run_len >= 5:
        return 'chronic_override'
    # Commitment-oriented: high dismiss rate
    if cs >= COMMITMENT_HI:
        return 'commitment_oriented'
    # Reward-oriented: very low dismiss rate
    if cs < COMMITMENT_LO:
        return 'reward_oriented'
    # Adaptive switcher: meaningful temporal change in open rate
    if abs(adapt) >= ADAPTATION_THRESHOLD:
        return 'adaptive_switcher'
    # Default: mixed / ambiguous
    return 'mixed'

pop['strategy'] = pop.apply(classify_strategy, axis=1)

print('Strategy distribution:')
print(pop['strategy'].value_counts())
print()
print('commitment_score by strategy:')
print(pop.groupby('strategy')['commitment_score'].agg(['mean','median','std']).round(3))


## Section 3: Condition Effect on Commitment

In [ ]:
COND_ORDER = ['control', 'default', 'reintervention']
COND_COLORS = {'control': '#4C72B0', 'default': '#DD8452', 'reintervention': '#55A868'}

cond_stats = (
    pop.groupby('study_condition')['commitment_score']
    .agg(n='count', mean='mean', median='median', std='std')
    .reindex(COND_ORDER).reset_index()
)
print('Commitment score by condition:')
print(cond_stats.to_string(index=False))
print()

# One-way ANOVA
groups = [pop.loc[pop['study_condition']==c,'commitment_score'].dropna().values
          for c in COND_ORDER]
f_stat, p_anova = stats.f_oneway(*groups)
print(f'ANOVA: F={f_stat:.3f}, p={p_anova:.4f}')
for c in ['default','reintervention']:
    t, p = stats.ttest_ind(
        pop.loc[pop['study_condition']=='control','commitment_score'].dropna(),
        pop.loc[pop['study_condition']==c,'commitment_score'].dropna(),
        equal_var=False
    )
    print(f'Welch t-test control vs {c}: t={t:.3f}, p={p:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('DS5 GaT — Commitment Score by Condition\n'
             'FLAG G1: commitment_score = rate_not_opened (behavioral proxy)', fontsize=10)

# Left: violin by condition
ax = axes[0]
data = [pop.loc[pop['study_condition']==c,'commitment_score'].dropna().values
        for c in COND_ORDER]
parts = ax.violinplot(data, positions=range(len(COND_ORDER)),
                      showmedians=True, showextrema=False)
for pc, c in zip(parts['bodies'], COND_ORDER):
    pc.set_facecolor(COND_COLORS[c]); pc.set_alpha(0.6)
parts['cmedians'].set_color('black')
ax.set_xticks(range(len(COND_ORDER)))
ax.set_xticklabels(COND_ORDER)
ax.set_ylabel('Commitment score (rate_not_opened)')
ax.set_title('Commitment score by condition')
for i, (vals, c) in enumerate(zip(data, COND_ORDER)):
    ax.text(i, np.nanpercentile(vals, 97),
            f'n={len(vals)}\nm={np.mean(vals):.3f}',
            ha='center', va='bottom', fontsize=8)

# Right: strategy distribution by condition (stacked bar)
ax = axes[1]
strat_by_cond = pd.crosstab(pop['study_condition'], pop['strategy'], normalize='index')
strat_by_cond = strat_by_cond.reindex(COND_ORDER)
strat_colors = {
    'commitment_oriented': '#27AE60',
    'reward_oriented':     '#E74C3C',
    'chronic_override':    '#8E44AD',
    'adaptive_switcher':   '#F39C12',
    'mixed':               '#95A5A6',
}
strat_order = ['commitment_oriented','reward_oriented','chronic_override',
               'adaptive_switcher','mixed']
bottom = np.zeros(len(COND_ORDER))
for s in strat_order:
    if s in strat_by_cond.columns:
        vals = strat_by_cond[s].values
        ax.bar(range(len(COND_ORDER)), vals, bottom=bottom,
               color=strat_colors[s], alpha=0.8, label=s.replace('_',' '))
        bottom += vals

ax.set_xticks(range(len(COND_ORDER)))
ax.set_xticklabels(COND_ORDER)
ax.set_ylabel('Proportion')
ax.set_title('Strategy distribution by condition')
ax.legend(fontsize=8, bbox_to_anchor=(1,1), loc='upper left')
ax.set_ylim(0, 1)

plt.tight_layout()
fp = FIGS / 'fig1_gat_commitment_by_condition.png'
plt.savefig(fp, dpi=150, bbox_inches='tight')
print(f'Saved: {fp}')
plt.show()


## Section 4: Phenotype Overlay — GaT strategies by DS5_01 Profile

In [ ]:
pop_pheno = pop[pop['profile_p1'].notna()].copy()

pheno_stats = (
    pop_pheno.groupby('profile_p1')['commitment_score']
    .agg(n='count', mean='mean', median='median', std='std')
    .reset_index()
)
print('Commitment score by DS5_01 phenotype:')
print(pheno_stats.to_string(index=False))
print()

# T-test A vs B
a_vals = pop_pheno.loc[pop_pheno['profile_p1']=='Profile A','commitment_score'].dropna()
b_vals = pop_pheno.loc[pop_pheno['profile_p1']=='Profile B','commitment_score'].dropna()
t_stat, p_val = stats.ttest_ind(a_vals, b_vals, equal_var=False)
print(f'Welch t-test Profile A vs B: t={t_stat:.3f}, p={p_val:.4f}')
d_cohen = (a_vals.mean() - b_vals.mean()) / np.sqrt(
    (a_vals.std()**2 + b_vals.std()**2) / 2)
print(f"Cohen's d = {d_cohen:.3f}")
print()

# Strategy by phenotype
print('Strategy distribution by phenotype:')
print(pd.crosstab(pop_pheno['profile_p1'], pop_pheno['strategy'], normalize='index').round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
PHENO_COLORS = {'Profile A': '#A20A7C', 'Profile B': '#7E57C2'}
fig.suptitle('DS5 GaT — Commitment and Strategy by Phenotype Profile', fontsize=10)

# Left: commitment by phenotype
ax = axes[0]
data_ph = [pop_pheno.loc[pop_pheno['profile_p1']==p,'commitment_score'].dropna().values
           for p in ['Profile A','Profile B']]
parts = ax.violinplot(data_ph, positions=[0,1], showmedians=True, showextrema=False)
for pc, p in zip(parts['bodies'], ['Profile A','Profile B']):
    pc.set_facecolor(PHENO_COLORS[p]); pc.set_alpha(0.7)
parts['cmedians'].set_color('black')
ax.set_xticks([0,1]); ax.set_xticklabels(['Profile A','Profile B'])
ax.set_ylabel('Commitment score (rate_not_opened)')
ax.set_title(f'Commitment by phenotype\n(t={t_stat:.2f}, p={p_val:.4f}, d={d_cohen:.3f})')
for i, (vals, p) in enumerate(zip(data_ph, ['Profile A','Profile B'])):
    ax.text(i, np.nanpercentile(vals, 97),
            f'n={len(vals)}\nm={np.mean(vals):.3f}',
            ha='center', va='bottom', fontsize=8)

# Right: strategy by phenotype (stacked bar)
ax = axes[1]
strat_by_pheno = pd.crosstab(pop_pheno['profile_p1'], pop_pheno['strategy'],
                              normalize='index').reindex(['Profile A','Profile B'])
bottom = np.zeros(2)
for s in strat_order:
    if s in strat_by_pheno.columns:
        vals = strat_by_pheno[s].values
        ax.bar([0,1], vals, bottom=bottom, color=strat_colors[s], alpha=0.8,
               label=s.replace('_',' '))
        bottom += vals
ax.set_xticks([0,1]); ax.set_xticklabels(['Profile A','Profile B'])
ax.set_ylabel('Proportion'); ax.set_title('Strategy by phenotype')
ax.legend(fontsize=8, bbox_to_anchor=(1,1), loc='upper left')
ax.set_ylim(0,1)

plt.tight_layout()
fp = FIGS / 'fig2_gat_commitment_by_phenotype.png'
plt.savefig(fp, dpi=150, bbox_inches='tight')
print(f'Saved: {fp}')
plt.show()


## Section 5: Adaptation — Temporal Change in Commitment

In [ ]:
# Adaptation: open_rate_change_calendar = late_third - early_third
# Negative value → open rate declining (commitment increasing over time = adaptive toward self-regulation)
# Positive value → open rate increasing (commitment declining = rebound)

pop['adaptation_direction'] = np.where(
    pop['open_rate_change_calendar'] < -ADAPTATION_THRESHOLD, 'adapting_toward_dismissal',
    np.where(pop['open_rate_change_calendar'] > ADAPTATION_THRESHOLD, 'rebounding_toward_open',
    'stable')
)

print('Adaptation direction distribution:')
print(pop['adaptation_direction'].value_counts())
print()
print('Adaptation by condition:')
print(pd.crosstab(pop['study_condition'], pop['adaptation_direction'],
                  normalize='index').round(3))
print()
print('Adaptation by phenotype:')
print(pd.crosstab(pop['profile_p1'], pop['adaptation_direction'],
                  normalize='index').round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle('DS5 GaT — Temporal Adaptation in Override Rate\n'
             '(open_rate_change_calendar = late 1/3 − early 1/3)', fontsize=10)

for c in COND_ORDER:
    sub = pop.loc[pop['study_condition']==c,'open_rate_change_calendar'].dropna()
    ax.hist(sub, bins=30, alpha=0.5, color=COND_COLORS[c], label=c, density=True)

ax.axvline(0, color='black', lw=1.5, ls='--', label='No change')
ax.axvline(-ADAPTATION_THRESHOLD, color='grey', lw=1, ls=':', alpha=0.7,
           label=f'±{ADAPTATION_THRESHOLD:.0%} threshold')
ax.axvline(ADAPTATION_THRESHOLD,  color='grey', lw=1, ls=':', alpha=0.7)
ax.set_xlabel('Change in open rate (late − early calendar third)')
ax.set_ylabel('Density')
ax.set_title('Distribution of adaptation direction by condition')
ax.legend()

plt.tight_layout()
fp = FIGS / 'fig3_gat_adaptation_by_condition.png'
plt.savefig(fp, dpi=150, bbox_inches='tight')
print(f'Saved: {fp}')
plt.show()


## Section 6: Export

In [ ]:
# Build export
export_cols = ['participant_id_canonical','study_condition',
               'commitment_score','strategy','adaptation_direction',
               'open_rate_change_calendar','max_consecutive_opens','mean_open_run_length',
               'p_open_after_dismiss','profile_p1']
available_export = [c for c in export_cols if c in pop.columns]
export = pop[available_export].copy()

out_csv = GAT_OUT / 'participant_gat_parameters.csv'
export.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}  ({len(export)} rows)')

# Model run summary
strat_counts = pop['strategy'].value_counts()
adapt_counts = pop['adaptation_direction'].value_counts()

md = f"""# DS5 GaT — Model Run Summary

**Run timestamp:** {MODEL_RUN_TS}
**Notebook:** notebooks/46_ds5_game_theory.ipynb
**Model:** Game Theory (GaT), Step 1 — descriptive commitment + strategy classification
**Dataset:** DS5 — German Adolescents RCT (N=573)

---

## FLAG decisions

| Flag | Decision |
|---|---|
| FLAG G1 | commitment_score = rate_not_opened (behavioral proxy, not utility) |
| FLAG G2 | Strategy thresholds descriptive (not equilibrium-derived) |
| FLAG G3 | No observed payoffs |
| FLAG G4 | Adaptation = calendar-third change, not sequential update |

## Population

- Total participants: {len(pop)}
- With phenotype (Profile A/B): {pop['profile_p1'].notna().sum()}

## Commitment score by condition

| Condition | N | Mean | Median |
|---|---|---|---|
{chr(10).join(f'| {r.study_condition} | {r.n} | {r["mean"]:.3f} | {r["median"]:.3f} |' for _, r in cond_stats.iterrows())}

ANOVA: F={f_stat:.3f}, p={p_anova:.4f}

## Strategy distribution

{chr(10).join(f'- {s}: {n} ({100*n/len(pop):.1f}%)' for s, n in strat_counts.items())}

## Commitment by phenotype

| Profile | N | Mean | Median | Cohen d vs A |
|---|---|---|---|---|
| Profile A | {len(a_vals)} | {a_vals.mean():.3f} | {a_vals.median():.3f} | — |
| Profile B | {len(b_vals)} | {b_vals.mean():.3f} | {b_vals.median():.3f} | {d_cohen:.3f} |

Welch t: t={t_stat:.3f}, p={p_val:.4f}

## Adaptation direction

{chr(10).join(f'- {s}: {n} ({100*n/len(pop):.1f}%)' for s, n in adapt_counts.items())}

## Output files

| File | Description |
|---|---|
| participant_gat_parameters.csv | Per-participant commitment score, strategy, adaptation |
| figures/fig1_gat_commitment_by_condition.png | Violin + strategy bar by condition |
| figures/fig2_gat_commitment_by_phenotype.png | Violin + strategy bar by phenotype |
| figures/fig3_gat_adaptation_by_condition.png | Temporal adaptation histogram |

## Known limitations

1. FLAG G1: commitment_score is a behavioral ratio — not a formal payoff utility measure
2. FLAG G2: Strategy thresholds (60%/30%) are interpretive; results are threshold-sensitive
3. FLAG G3: No payoff data directly observed; strategy interpretation requires theoretical framing
4. The open/dismiss decision is treated as one-dimensional; substitution (close) behavior not separately modeled
5. No sequential strategy-state model (Bayesian updating); adaptation captured as aggregate calendar-period change only
"""

out_md = GAT_OUT / 'model_run_summary.md'
out_md.write_text(md)
print(f'Saved: {out_md}')


## Section 7: Summary

In [ ]:
print('=' * 60)
print('DS5 GaT — Model Run Summary')
print('=' * 60)
print()
print(f'Commitment score (all N={len(pop)}):')
print(f'  Mean: {pop["commitment_score"].mean():.3f}  Median: {pop["commitment_score"].median():.3f}')
print()
print('Strategy distribution:')
for s, n in pop['strategy'].value_counts().items():
    print(f'  {s:30s}: {n:3d} ({100*n/len(pop):.1f}%)')
print()
print(f'ANOVA commitment by condition: F={f_stat:.3f}, p={p_anova:.4f}')
print()
print(f'Profile A commitment: M={a_vals.mean():.3f} (N={len(a_vals)})')
print(f'Profile B commitment: M={b_vals.mean():.3f} (N={len(b_vals)})')
print(f"Phenotype difference: t={t_stat:.3f}, p={p_val:.4f}, Cohen's d={d_cohen:.3f}")
print()
print('Figures: fig1, fig2, fig3 saved to:', FIGS)
print('Data:    participant_gat_parameters.csv saved to:', GAT_OUT)
